# Decision-boundary demos — FGSM and PGD on toy 2-D data

The presentation notebooks: tiny 2-D classification problems where the decision boundary can be drawn, so FGSM and PGD perturbations can be *seen* — first on linearly separable data with a simple and then a non-linear model, then on non-linearly separable data.

**Where this fits:** this section is the study material behind the thesis work — the tutorial chapters and side experiments it grew out of. It is coursework, kept for completeness and clearly separate from the pipeline in sections 01–06; the image-domain experiments here are the image counterparts of the feature-space work in `03-attacks/`.

*Consolidated from `ML_presentation.ipynb` and `ML_presentation2.ipynb` (originally in `Adversarials`). Outputs cleared.*

## Part A — Linearly separable data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Generating synthetic data for two classes
torch.manual_seed(42)
class_1 = torch.randn(100, 2) + 1.5  # Class 1 centered around (1.5, 1.5)
class_2 = torch.randn(100, 2) - 1.5  # Class 2 centered around (-1.5, -1.5)

# Labels for the classes (0 and 1)
labels_class_1 = torch.zeros(100, 1)
labels_class_2 = torch.ones(100, 1)

# Combining the data and labels
data = torch.cat((class_1, class_2), dim=0)
labels = torch.cat((labels_class_1, labels_class_2), dim=0)

In [ ]:
data.shape

In [ ]:
plt.scatter(data[:,0], data[:,1], c=labels.squeeze())

In [ ]:
# Define a simple neural network model
class SimpleClassifier(nn.Module):
    def __init__(self):
        super(SimpleClassifier, self).__init__()
        self.fc = nn.Linear(2, 1)

    def forward(self, x):
        x = self.fc(x)
        return x

# Initialize the model, loss function, and optimizer
model = SimpleClassifier()
criterion = nn.BCEWithLogitsLoss()  # Binary Cross Entropy Loss
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Training the model
epochs = 1000
for epoch in range(epochs):
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np


# Retrieve the parameters (weights and bias) from the trained model
trained_params = model.state_dict()
weights = trained_params['fc.weight'].detach().numpy()
bias = trained_params['fc.bias'].detach().numpy()

# Scatter plot the data points
plt.scatter(class_1[:, 0], class_1[:, 1], label='Class 1')
plt.scatter(class_2[:, 0], class_2[:, 1], label='Class 2')

# Plot the decision boundary line
x_vals = np.array([-4, 4])  # Choose x values within the range of your data
y_vals = -(weights[0][0] * x_vals + bias) / weights[0][1]
plt.plot(x_vals, y_vals, color='red', label='Decision Boundary')

plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Decision Boundary of the Classifier')
plt.legend()
plt.show()

In [ ]:
# Plotting the decision boundary
def plot_decision_boundary(model, data, labels):
    plt.scatter(data[:, 0], data[:, 1], c=labels.squeeze(), cmap='viridis', edgecolor='k')
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    xx, yy = torch.meshgrid(torch.linspace(xlim[0], xlim[1], 100),
                            torch.linspace(ylim[0], ylim[1], 100))
    input_grid = torch.cat([xx.reshape(-1, 1), yy.reshape(-1, 1)], dim=1)
    Z = model(input_grid).detach().numpy().reshape(xx.shape)

    plt.contour(xx, yy, Z, levels=[0.], colors='black', linestyles='dashed')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title('Classification with PyTorch')
    plt.show()

# Visualizing the decision boundary
plot_decision_boundary(model, data, labels)

In [ ]:
def fgsm(model, X, y, epsilon=0.1):
    """ Construct FGSM adversarial examples on the examples X"""
    delta = torch.zeros_like(X, requires_grad=True)
    loss = nn.BCEWithLogitsLoss()(model(X + delta), y)
    loss.backward()
    return epsilon * delta.grad.detach().sign()

In [ ]:
# Define a sample input data and its label
input_data = torch.tensor([[-2., -2.]], requires_grad=True)  # Replace with your input data
true_label = torch.tensor([[1.0]])  # Replace with the true label (target label)

In [ ]:
x = data[:1]
y = labels[:1]

fgsm(model, data[:1], labels[:1], epsilon=0.1)

In [ ]:
x = data[:1]
y = labels[:1]

delta = fgsm(model, data[:1], labels[:1], epsilon=1)

print(model(x+delta))
print(model(x))

In [ ]:
x2 = x+delta

In [ ]:
plt.scatter(x[0][0],x[0][1])
plt.scatter(x2[0][0],x2[0][1])

In [ ]:
x = data
y = labels

delta = fgsm(model, data, labels, epsilon=0.3)

In [ ]:
# Plotting the decision boundary
def plot_decision_boundary(model, data, labels):
    plt.figure(figsize=(15, 12))
    plt.scatter(x[:, 0], x[:, 1], c=labels.squeeze())
    plt.scatter(x2[:, 0], x2[:, 1], c=labels.squeeze(), cmap='viridis', edgecolor='y')
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    xx, yy = torch.meshgrid(torch.linspace(xlim[0], xlim[1], 100),
                            torch.linspace(ylim[0], ylim[1], 100))
    input_grid = torch.cat([xx.reshape(-1, 1), yy.reshape(-1, 1)], dim=1)
    Z = model(input_grid).detach().numpy().reshape(xx.shape)

    plt.contour(xx, yy, Z, levels=[0.], colors='black', linestyles='dashed')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title('Classification with PyTorch')
    plt.show()

# Visualizing the decision boundary
plot_decision_boundary(model, data, labels)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# Generating nonlinearly separable synthetic data for two classes
torch.manual_seed(42)

# Generating class 1 data in a circular pattern
theta = torch.linspace(0, 4 * np.pi, 100)
r = 5 + torch.randn(100) * 1.5  # Radius with some noise
class_1_x = r * torch.cos(theta)
class_1_y = r * torch.sin(theta)
class_1 = torch.stack((class_1_x, class_1_y), dim=1)

# Generating class 2 data in a circular pattern with different parameters
theta = torch.linspace(0, 2 * np.pi, 100)
r = 8 + torch.randn(100) * 1.5  # Radius with some noise
class_2_x = r * torch.cos(theta)
class_2_y = r * torch.sin(theta)
class_2 = torch.stack((class_2_x, class_2_y), dim=1)

# Labels for the classes (0 and 1)
labels_class_1 = torch.zeros(100, 1)
labels_class_2 = torch.ones(100, 1)

# Combining the data and labels
data = torch.cat((class_1, class_2), dim=0)
labels = torch.cat((labels_class_1, labels_class_2), dim=0)

# Visualize the generated data
plt.scatter(class_1[:, 0], class_1[:, 1], label='Class 1')
plt.scatter(class_2[:, 0], class_2[:, 1], label='Class 2')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Nonlinearly Separable Data')
plt.legend()
plt.show()

In [ ]:
# Define a neural network model with non-linear activations
class NonlinearClassifier(nn.Module):
    def __init__(self):
        super(NonlinearClassifier, self).__init__()
        self.fc1 = nn.Linear(2, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x


# Initialize the model, loss function, and optimizer
model = NonlinearClassifier()
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training the model
epochs = 1000
for epoch in range(epochs):
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# No boundary visualization for this nonlinear case as it's harder to represent in 2D.

In [ ]:
# Create a grid of points to represent the entire space
x_min, x_max = data[:, 0].min() - 1, data[:, 0].max() + 1
y_min, y_max = data[:, 1].min() - 1, data[:, 1].max() + 1
xx, yy = torch.meshgrid(torch.linspace(x_min, x_max, 1000), torch.linspace(y_min, y_max, 1000))
grid_data = torch.stack([xx.flatten(), yy.flatten()], dim=1)

# Predict classes for each point in the grid
with torch.no_grad():
    model.eval()
    predictions = model(grid_data).numpy().reshape(xx.shape)

# Threshold the predictions to get class labels (0 or 1)
threshold = 0.5
predictions = (predictions > threshold).astype(np.int)

# Plot decision boundary as lines
plt.contour(xx, yy, predictions, levels=[0.5], colors='black')
plt.scatter(class_1[:, 0], class_1[:, 1], label='Class 1')
plt.scatter(class_2[:, 0], class_2[:, 1], label='Class 2')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Decision Boundary of Nonlinear Classifier')
plt.legend()
plt.show()

In [ ]:
def pgd(model, X, y, epsilon, alpha, num_iter):
    """ Construct FGSM adversarial examples on the examples X"""
    delta = torch.zeros_like(X, requires_grad=True)
    for t in range(num_iter):
        loss = nn.BCEWithLogitsLoss()(model(X + delta), y)
        loss.backward()
        delta.data = (delta + X.shape[0]*alpha*delta.grad.data).clamp(-epsilon,epsilon)
        delta.grad.zero_()
    return delta.detach()

In [ ]:
# Original data
x = data
y = labels

# Applying perturbation using FGSM
delta = pgd(model, x, y, 0.8, 0.1, 1000)
x2 = x + delta

In [ ]:
# Plotting the decision boundary
def plot_decision_boundary(model, data, labels):
    plt.scatter(data[:, 0], data[:, 1], c=labels.squeeze(), cmap='viridis', edgecolor='k')
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    xx, yy = torch.meshgrid(torch.linspace(xlim[0], xlim[1], 100),
                            torch.linspace(ylim[0], ylim[1], 100))
    input_grid = torch.cat([xx.reshape(-1, 1), yy.reshape(-1, 1)], dim=1)
    Z = model(input_grid).detach().numpy().reshape(xx.shape)

    plt.contour(xx, yy, Z, levels=[0.0000001], colors='black', linestyles='dashed')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title('Classification with PyTorch')
    plt.show()

# Visualizing the decision boundary
plot_decision_boundary(model, data, labels)

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# Assuming you have defined your model

# Generate a grid of input values
x_min, x_max = -10, 10
y_min, y_max = -10, 10
num_points = 100
xx, yy = torch.meshgrid(torch.linspace(x_min, x_max, num_points), torch.linspace(y_min, y_max, num_points))
input_grid = torch.stack([xx.flatten(), yy.flatten()], dim=1)

# Compute model predictions for each point in the input space
with torch.no_grad():
    model.eval()
    predictions = model(input_grid)
    predicted_labels = (predictions > 0.5).float()  # Assuming binary classification, adjust threshold if needed

predicted_labels = predictions.reshape(xx.shape)

# Plot the contour plot of predicted labels in the input space
plt.figure(figsize=(20, 20))
contour = plt.contourf(xx, yy, predicted_labels, levels=100, cmap='coolwarm')
plt.colorbar(contour)
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Contour Plot of Predicted Labels in Input Space')
plt.show()

## Part B — Non-linearly separable data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Generating nonlinearly separable synthetic data for two classes
torch.manual_seed(42)

n =150

# Generating class 1 data in a circular pattern
theta = torch.linspace(0, 4 * np.pi, n)
r = 5 + torch.randn(n) * 1.5  # Radius with some noise
class_1_x = r * torch.cos(theta)
class_1_y = r * torch.sin(theta)
class_1 = torch.stack((class_1_x, class_1_y), dim=1)

# Generating class 2 data in a circular pattern with different parameters
theta = torch.linspace(0, 2 * np.pi, n)
r = 8 + torch.randn(n) * 1.5  # Radius with some noise
class_2_x = r * torch.cos(theta)
class_2_y = r * torch.sin(theta)
class_2 = torch.stack((class_2_x, class_2_y), dim=1)

# Labels for the classes (0 and 1)
labels_class_1 = torch.zeros(n, 1)
labels_class_2 = torch.ones(n, 1)

# Combining the data and labels
data = torch.cat((class_1, class_2), dim=0)
labels = torch.cat((labels_class_1, labels_class_2), dim=0)

# Visualize the generated data
plt.scatter(class_1[:, 0], class_1[:, 1], label='Class 1')
plt.scatter(class_2[:, 0], class_2[:, 1], label='Class 2')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Nonlinearly Separable Data')
plt.legend()
plt.show()

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Define a neural network model with non-linear activations
class NonlinearClassifier(nn.Module):
    def __init__(self):
        super(NonlinearClassifier, self).__init__()
        self.fc1 = nn.Linear(2, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x


# Initialize the model, loss function, and optimizer
model = NonlinearClassifier()
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training the model
epochs = 1000
for epoch in range(epochs):
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# No boundary visualization for this nonlinear case as it's harder to represent in 2D.

In [ ]:
# Generate a grid of input values
x_min, x_max = -10, 10
y_min, y_max = -10, 10
num_points = 100
xx, yy = torch.meshgrid(torch.linspace(x_min, x_max, num_points), torch.linspace(y_min, y_max, num_points))
input_grid = torch.stack([xx.flatten(), yy.flatten()], dim=1)

# Compute model predictions for each point in the input space
with torch.no_grad():
    model.eval()
    predictions = model(input_grid)
    predicted_labels = (predictions > 0.5).float()  # Assuming binary classification, adjust threshold if needed

predicted_labels = predictions.reshape(xx.shape)

# Plot the contour plot of predicted labels in the input space
plt.figure(figsize=(20, 20))
contour = plt.contourf(xx, yy, predicted_labels, levels=100, cmap='coolwarm')
plt.colorbar(contour)
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Contour Plot of Predicted Labels in Input Space')
plt.show()

In [ ]:
# Original data
x = data
y = labels

# Applying perturbation using FGSM
delta = fgsm(model, data, labels, epsilon=0.5)

# Filter perturbed samples that cause a change in label
changed_indices = ((model(x + delta).detach().squeeze() > 0.5) != (model(x).detach().squeeze() > 0.5))
changed_x = x[changed_indices]
changed_x2 = (x + delta)[changed_indices]

not_changed_x = x[~changed_indices]
not_changed_x2 = (x + delta)[~changed_indices]

# Plotting the route of change for perturbed samples that cause a label change
plt.figure(figsize=(20, 20))
for i in range(len(changed_x)):
    plt.plot([changed_x[i, 0], changed_x2[i, 0]], [changed_x[i, 1], changed_x2[i, 1]], 'r-', alpha=0.5)

# Plotting original and perturbed data points
plt.scatter(x[:, 0], x[:, 1], c=labels.squeeze(), marker='o', label='Original Data')
plt.scatter(changed_x2[:, 0], changed_x2[:, 1], c=labels[changed_indices].squeeze(), marker='x', label='Perturbed Data')

# Create a grid of points to represent the entire space
x_min, x_max = data[:, 0].min() - 1, data[:, 0].max() + 1
y_min, y_max = data[:, 1].min() - 1, data[:, 1].max() + 1
xx, yy = torch.meshgrid(torch.linspace(x_min, x_max, 1000), torch.linspace(y_min, y_max, 1000))
grid_data = torch.stack([xx.flatten(), yy.flatten()], dim=1)

# Predict classes for each point in the grid
with torch.no_grad():
    model.eval()
    predictions = model(grid_data).numpy().reshape(xx.shape)

# Threshold the predictions to get class labels (0 or 1)
threshold = 0.5
predictions = (predictions > threshold).astype(np.int)

# Plot decision boundary as lines
plt.contour(xx, yy, predictions, levels=[0.5], colors='black')

In [ ]:
def pgd(model, X, y, epsilon, alpha, num_iter):
    """ Construct PGD adversarial examples on the examples X"""
    delta = torch.zeros_like(X, requires_grad=True)
    perturbed_x_list = [X.clone().detach().numpy()]  # List to store intermediate perturbed x values

    for t in range(num_iter):
        loss = nn.BCEWithLogitsLoss()(model(X + delta), y)
        loss.backward()

        # Update delta and clamp within epsilon
        delta.data = (delta + alpha * delta.grad.sign()).clamp(-epsilon, epsilon)
        delta.grad.zero_()

        perturbed_x = (X + delta).clone().detach().numpy()
        perturbed_x_list.append(perturbed_x)

    return perturbed_x_list

In [ ]:
# Get perturbed x values from PGD attack
perturbed_x_list = pgd(model, x, y, epsilon=1., alpha=0.05, num_iter=50)

# Plotting the route of perturbed x with connecting lines
plt.figure(figsize=(20, 20))
for i in range(len(perturbed_x_list) - 1):
    plt.plot([perturbed_x_list[i][:, 0], perturbed_x_list[i + 1][:, 0]],
             [perturbed_x_list[i][:, 1], perturbed_x_list[i + 1][:, 1]], 'b-', alpha=0.5)
    #plt.scatter(perturbed_x_list[i][:, 0], perturbed_x_list[i][:, 1], alpha=0.5)

plt.scatter(perturbed_x_list[-1][:, 0], perturbed_x_list[-1][:, 1], label='Final Perturbed x', c='r')
plt.scatter(x[:, 0], x[:, 1], c=y.squeeze(), cmap='viridis', label='Original Data')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Route of Perturbed x during PGD Attack')
plt.legend()
plt.show()

In [ ]:
y = torch.tensor([[1.],[1.],[1.],[1.],[1.]])
x = torch.tensor([[2.5,7.5],[-1.,-11.],[2.5,6.]])

In [ ]:
# Assuming you have your data and computations from the provided code

fig, ax = plt.subplots(figsize=(20, 20))  # Create a single subplot

# Original data
x = torch.tensor([[5,-7.5],[-2.5,7.5],[-2.2,-10.2],[8.,1.],[0.,7.5],[-4.5,3.5]])
y = torch.tensor([[1.],[1.],[1.],[1.],[1.],[0.]])

# Applying perturbation using FGSM
delta = fgsm(model, x, y, epsilon=1.5)

# Filter perturbed samples that cause a change in label
x2 = (x + delta)

# Create a grid of points to represent the entire space
x_min, x_max = data[:, 0].min() - 1, data[:, 0].max() + 1
y_min, y_max = data[:, 1].min() - 1, data[:, 1].max() + 1
xx, yy = torch.meshgrid(torch.linspace(x_min, x_max, 100), torch.linspace(y_min, y_max, 100))
grid_data = torch.stack([xx.flatten(), yy.flatten()], dim=1)

# Compute model predictions for each point in the input space
with torch.no_grad():
    model.eval()
    predictions = model(grid_data)

predicted_labels = (predictions > 0.5).float()  # Assuming binary classification, adjust threshold if needed
predicted_labels = predicted_labels.reshape(xx.shape)

# Plot the contour plot of predicted labels in the input space
contour = ax.contourf(xx, yy, predictions.reshape(xx.shape) ,levels=np.linspace(0,1,100) , cmap='coolwarm')
plt.colorbar(contour, ax=ax)
ax.set_xlabel('X-axis')
ax.set_ylabel('Y-axis')
ax.set_title('Contour Plot of Predicted Labels with Route of Perturbed Samples')

# Plotting the route of change for perturbed samples causing a label change
for i in range(len(x)):
    ax.plot([x[i, 0], x2[i, 0]], [x[i, 1], x2[i, 1]], 'r-', alpha=0.5)

# Plotting original and perturbed data points
ax.scatter(x[:, 0], x[:, 1], c=y.squeeze(), marker='o', label='Original Data', s=10)
ax.scatter(x2[:, 0], x2[:, 1], c=y.squeeze(), marker='x', label='Perturbed Data', s=20)


# Get perturbed x values from PGD attack
perturbed_x_list = pgd(model, x, y, epsilon=1.5, alpha=0.1, num_iter=50)

for i in range(len(perturbed_x_list) - 1):
    plt.plot([perturbed_x_list[i][:, 0], perturbed_x_list[i + 1][:, 0]],
             [perturbed_x_list[i][:, 1], perturbed_x_list[i + 1][:, 1]], 'b-', alpha=0.5)
    #plt.scatter(perturbed_x_list[i][:, 0], perturbed_x_list[i][:, 1], alpha=0.5)

ax.scatter(perturbed_x_list[-1][:, 0], perturbed_x_list[-1][:, 1], label='Final Perturbed x', c='r')
ax.scatter(x[:, 0], x[:, 1], c=y.squeeze(), cmap='viridis', label='Original Data')


ax.legend()

plt.show()